<a href="https://colab.research.google.com/github/s-rafia/voice-controlled-design-tool/blob/main/notebooks/train_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/Projects/Voice_Controlled_Design_Tool/Data'

import os
print(os.listdir(DATA_DIR))

Mounted at /content/drive
['commands.csv', 'hard_eval.csv']


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(f'{DATA_DIR}/commands.csv')
print('rows:', len(df), '| labels:', df['label'].nunique())

# the 30 command names, in a fixed alphabetical order
labels = sorted(df['label'].unique())

# build the two lookup tables: name -> number, and number -> name
label2id = {}
id2label = {}
for i in range(len(labels)):
    label2id[labels[i]] = i
    id2label[i] = labels[i]

# add a column holding each row's label as a number
df['label_id'] = df['label'].map(label2id)

# split into 70% train, then cut the remaining 30% in half
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42
)

print('train:', len(train_df), 'rows')
print('validation:', len(val_df), 'rows')
print('test:', len(test_df), 'rows')

rows: 544 | labels: 30
train: 380 rows
validation: 82 rows
test: 82 rows


In [5]:
from datasets import Dataset
from transformers import AutoTokenizer

model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['phrase'], truncation=True, padding='max_length', max_length=32)

def to_dataset(frame):
    frame = frame[['phrase', 'label_id']]
    frame = frame.rename(columns={'label_id': 'label'})
    frame = frame.reset_index(drop=True)
    dataset = Dataset.from_pandas(frame)
    dataset = dataset.map(tokenize, batched=True)
    return dataset

train_dataset = to_dataset(train_df)
val_dataset = to_dataset(val_df)
test_dataset = to_dataset(test_df)


print(train_dataset)
print(train_dataset[0]['phrase'], '->', id2label[train_dataset[0]['label']])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/380 [00:00<?, ? examples/s]

Map:   0%|          | 0/82 [00:00<?, ? examples/s]

Map:   0%|          | 0/82 [00:00<?, ? examples/s]

Dataset({
    features: ['phrase', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 380
})
distribute these evenly -> DISTRIBUTE


In [6]:
import numpy as np
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, refs = eval_pred
    preds = np.argmax(logits, axis=1)
    accuracy = accuracy_score(refs, preds)
    macro_f1 = f1_score(refs, preds, average='macro')
    return {'accuracy': accuracy, 'macro_f1': macro_f1}

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=40,
    per_device_train_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=20,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,3.380985,3.291261,0.182927,0.097899
2,3.213148,3.006631,0.463415,0.411407
3,2.940304,2.675519,0.573171,0.531039
4,2.618183,2.336065,0.682927,0.653333
5,1.961660,2.007158,0.756098,0.726190
6,1.652775,1.721415,0.780488,0.764444
7,1.388128,1.492848,0.792683,0.777434
8,1.111061,1.254180,0.804878,0.786111
9,0.896520,1.090382,0.792683,0.778175
10,0.572854,0.950100,0.829268,0.822593


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=576, training_loss=0.8626789304738244, metrics={'train_runtime': 184.4838, 'train_samples_per_second': 82.392, 'train_steps_per_second': 5.204, 'total_flos': 75544120627200.0, 'train_loss': 0.8626789304738244, 'epoch': 24.0})

In [8]:
test_results = trainer.evaluate(test_dataset)
print(test_results)
print()
print('Test accuracy:', round(test_results['eval_accuracy'] * 100, 1), '%')
print('Test macro F1:', round(test_results['eval_macro_f1'] * 100, 1), '%')

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.032488,0.434908,24,0.853659,0.858095


{'eval_loss': 0.43490755558013916, 'eval_accuracy': 0.8536585365853658, 'eval_macro_f1': 0.8580952380952381}

Test accuracy: 85.4 %
Test macro F1: 85.8 %


In [9]:
hard_df = pd.read_csv(f'{DATA_DIR}/hard_eval.csv')
hard_df['label_id'] = hard_df['label'].map(label2id)
hard_dataset = to_dataset(hard_df)

hard_results = trainer.evaluate(hard_dataset)

print('Held-out accuracy:', round(hard_results['eval_accuracy'] * 100, 1), '%')
print('Held-out macro F1:', round(hard_results['eval_macro_f1'] * 100, 1), '%')

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.032488,0.341098,24,0.933333,0.928889


Held-out accuracy: 93.3 %
Held-out macro F1: 92.9 %


In [10]:
from sklearn.metrics import classification_report
import torch

def predict_frame(frame):
    dataset = to_dataset(frame)
    predictions = trainer.predict(dataset)
    logits = predictions.predictions
    probabilities = torch.softmax(torch.tensor(logits), dim=1).numpy()
    predicted_ids = probabilities.argmax(axis=1)
    confidences = probabilities.max(axis=1)
    return predicted_ids, confidences

predicted_ids, confidences = predict_frame(hard_df)
true_ids = hard_df['label_id'].values

report = classification_report(
    true_ids, predicted_ids,
    labels=range(len(labels)), target_names=labels, zero_division=0
)
print(report)

print()
print('MISTAKES ON HELD-OUT SET')
print('-' * 78)

wrong_count = 0
for phrase, true_id, pred_id, conf in zip(hard_df['phrase'], true_ids, predicted_ids, confidences):
    if true_id != pred_id:
        wrong_count += 1
        true_label = id2label[true_id]
        pred_label = id2label[pred_id]
        print(f'{phrase:42s} true={true_label:18s} pred={pred_label:18s} conf={conf:.2f}')

print('-' * 78)
print(wrong_count, 'wrong out of', len(hard_df))

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

                   precision    recall  f1-score   support

 ADD_TO_SELECTION       1.00      0.50      0.67         2
            ALIGN       1.00      1.00      1.00         2
    BRING_FORWARD       1.00      1.00      1.00         2
     CREATE_SHAPE       1.00      1.00      1.00         2
      CREATE_TEXT       1.00      1.00      1.00         2
           DELETE       0.67      1.00      0.80         2
         DESELECT       1.00      0.50      0.67         2
       DISTRIBUTE       1.00      1.00      1.00         2
        DUPLICATE       1.00      1.00      1.00         2
             FLIP       1.00      1.00      1.00         2
            GROUP       1.00      1.00      1.00         2
       LOCK_LAYER       0.67      1.00      0.80         2
      MOVE_OBJECT       1.00      1.00      1.00         2
              PAN       1.00      1.00      1.00         2
             REDO       1.00      1.00      1.00         2
           RESIZE       1.00      1.00      1.00       

In [11]:
correct_confidences = []
wrong_confidences = []

for true_id, pred_id, conf in zip(true_ids, predicted_ids, confidences):
    if true_id == pred_id:
        correct_confidences.append(conf)
    else:
        wrong_confidences.append(conf)

correct_confidences.sort()

print('CORRECT:', len(correct_confidences), 'predictions')
print('  lowest confidence:', round(min(correct_confidences), 2))
print('  ten lowest:')
for c in correct_confidences[:10]:
    print('   ', round(c, 2))

print()
print('WRONG:', len(wrong_confidences), 'predictions')
print('  highest confidence:', round(max(wrong_confidences), 2))

print()
print('THRESHOLD SWEEP')
for threshold in [0.40, 0.50, 0.60, 0.70]:
    blocked_wrong = 0
    for c in wrong_confidences:
        if c < threshold:
            blocked_wrong = blocked_wrong + 1
    blocked_correct = 0
    for c in correct_confidences:
        if c < threshold:
            blocked_correct = blocked_correct + 1
    print('  threshold', threshold, ': blocks', blocked_wrong, 'of', len(wrong_confidences),
          'wrong and', blocked_correct, 'of', len(correct_confidences), 'correct')

CORRECT: 56 predictions
  lowest confidence: 0.26
  ten lowest:
    0.26
    0.48
    0.51
    0.57
    0.64
    0.78
    0.78
    0.82
    0.87
    0.9

WRONG: 4 predictions
  highest confidence: 0.58

THRESHOLD SWEEP
  threshold 0.4 : blocks 1 of 4 wrong and 1 of 56 correct
  threshold 0.5 : blocks 1 of 4 wrong and 2 of 56 correct
  threshold 0.6 : blocks 4 of 4 wrong and 4 of 56 correct
  threshold 0.7 : blocks 4 of 4 wrong and 5 of 56 correct


In [13]:
THRESHOLD = 0.60

def predict(phrase):
    inputs = tokenizer(phrase, return_tensors='pt', truncation=True,
                       padding='max_length', max_length=32)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probabilities = torch.softmax(logits, dim=1)[0]
    confidence, index = probabilities.max(0)
    confidence = confidence.item()
    intent = id2label[index.item()]
    return intent, confidence

while True:
    text = input('Say something (or "quit"): ')
    if text.strip().lower() == 'quit':
        break
    intent, confidence = predict(text.strip().lower())
    if confidence < THRESHOLD:
        print('   not confident enough:', round(confidence, 2), '- best guess was', intent)
    else:
        print('  ', intent, '- confidence', round(confidence, 2))

Say something (or "quit"): move the box to left
   MOVE_OBJECT - confidence 0.96
Say something (or "quit"): align all to left
   ALIGN - confidence 0.97
Say something (or "quit"): zoom out a little
   ZOOM_OUT - confidence 0.96
Say something (or "quit"): I need to see the entire screen
   not confident enough: 0.35 - best guess was ZOOM_IN
Say something (or "quit"): select all the red ones
   SELECT_BY_COLOR - confidence 0.98
Say something (or "quit"): make it lighter
   SET_OPACITY - confidence 0.71
Say something (or "quit"): quit


In [14]:
import json

SAVE_DIR = '/content/drive/MyDrive/Projects/Voice_Controlled_Design_Tool/model'

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

label_maps = {'labels': labels, 'label2id': label2id, 'id2label': id2label}
with open(SAVE_DIR + '/label_maps.json', 'w') as f:
    json.dump(label_maps, f, indent=2)

results = {
    'test_accuracy': test_results['eval_accuracy'],
    'test_macro_f1': test_results['eval_macro_f1'],
    'heldout_accuracy': hard_results['eval_accuracy'],
    'heldout_macro_f1': hard_results['eval_macro_f1'],
    'best_epoch': 12,
    'stopped_at_epoch': 17,
    'confidence_threshold': 0.40,
    'n_train': len(train_df),
    'n_val': len(val_df),
    'n_test': len(test_df),
    'n_heldout': len(hard_df),
    'n_labels': len(labels),
}
with open(SAVE_DIR + '/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('saved to', SAVE_DIR)
print(os.listdir(SAVE_DIR))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to /content/drive/MyDrive/Projects/Voice_Controlled_Design_Tool/model
['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'label_maps.json', 'results.json']
